In [1]:
from google.colab import files
uploaded = files.upload()

Saving customer_support_text_classification.csv to customer_support_text_classification.csv


# Task 1:Dataset understanding

In [2]:
import pandas as pd

df = pd.read_csv('customer_support_text_classification.csv')

# Number of records
print("Number of records:", df.shape[0])

# Target labels/classes
print("\nTarget classes:", df['sentiment_label'].unique())

# Sample text records
print("\nSample records:")
print(df[['customer_message', 'sentiment_label']].head(5))

# Average text length
print("\nAverage text length (words):", round(df['word_count'].mean(), 2))

# Class distribution
print("\nClass distribution:")
print(df['sentiment_label'].value_counts())

Number of records: 1500

Target classes: ['neutral' 'positive' 'negative']

Sample records:
                                    customer_message sentiment_label
0  I need information about the payment process. ...         neutral
1      I need information about the payment process.         neutral
2  The refund process was fast and convenient. I ...        positive
3  My refund is still pending and this experience...        negative
4   Please tell me how to update my account details.         neutral

Average text length (words): 12.72

Class distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64


# Task 2 : Text Preprocessing


In [3]:
import re
import string
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercasing
    text = text.lower()
    # Removing special characters and symbols
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenization
    tokens = text.split()
    # Removing stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Joining back
    return ' '.join(tokens)

df['clean_text'] = df['customer_message'].apply(clean_text)

# Check before and after
print("Original:", df['customer_message'][0])
print("Cleaned: ", df['clean_text'][0])
print("\nCleaned sample:")
print(df[['customer_message', 'clean_text']].head(3))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Original: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Cleaned:  need information payment process ticket number please respond soon possible

Cleaned sample:
                                    customer_message  \
0  I need information about the payment process. ...   
1      I need information about the payment process.   
2  The refund process was fast and convenient. I ...   

                                          clean_text  
0  need information payment process ticket number...  
1                   need information payment process  
2  refund process fast convenient appreciate quic...  


# Task 3: Text Vectorization

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Bag of Words
bow = CountVectorizer(max_features=1000)
X_bow = bow.fit_transform(df['clean_text'])

# TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

print("Bag of Words shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)

print("\nSample BoW feature names (first 10):")
print(bow.get_feature_names_out()[:10])

print("\nSample TF-IDF feature names (first 10):")
print(tfidf.get_feature_names_out()[:10])

Bag of Words shape: (1500, 146)
TF-IDF shape: (1500, 146)

Sample BoW feature names (first 10):
['account' 'activate' 'ago' 'analytics' 'app' 'appreciate' 'arrived'
 'assigned' 'available' 'bad']

Sample TF-IDF feature names (first 10):
['account' 'activate' 'ago' 'analytics' 'app' 'appreciate' 'arrived'
 'assigned' 'available' 'bad']


## Why text must be converted into vectors?

Machine learning models can only understand numbers, not words.
So we convert text into numerical vectors so the model can process it.

- Bag of Words (BoW): counts how many times each word appears
- TF-IDF: gives more importance to rare/meaningful words and less
  to common words like "the", "is"

Without vectorization, the model has no way to compare or
learn patterns from text data.

# Task 4: Baseline Model

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['sentiment_label'])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

# Train model
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



# Task 5: Sequence Model or Conceptual Architecture

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Settings
MAX_WORDS = 5000
MAX_LEN = 50

# Tokenize
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df['clean_text'])
X_seq = tokenizer.texts_to_sequences(df['clean_text'])
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN)

# Encode labels
y_cat = to_categorical(y, num_classes=3)

# Train/test split
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pad, y_cat, test_size=0.2, random_state=42
)

# Build LSTM
model_lstm = Sequential([
    Embedding(MAX_WORDS, 64, input_length=MAX_LEN),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model_lstm.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

model_lstm.summary()

# Train
model_lstm.fit(X_train2, y_train2,
               epochs=5,
               batch_size=32,
               validation_split=0.1)

# Evaluate
loss, acc = model_lstm.evaluate(X_test2, y_test2)
print(f"\nLSTM Test Accuracy: {round(acc, 4)}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.7657 - loss: 1.0145 - val_accuracy: 0.7000 - val_loss: 0.7023
Epoch 2/5
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9620 - loss: 0.2699 - val_accuracy: 1.0000 - val_loss: 0.0094
Epoch 3/5
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 1.0000 - loss: 0.0080 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 4/5
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 1.0000 - loss: 0.0021 - val_accuracy: 1.0000 - val_loss: 5.1019e-04
Epoch 5/5
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 3.2372e-04
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 3.0794e-04

LSTM Test Accuracy: 1.0


# Task 6: Attention and Transformer Reflection

## Task 6: Attention and Transformer Reflection

### Why RNNs struggle with long-term dependencies
RNNs process text word by word and pass information through a
hidden state. As the sentence gets longer, earlier information
gets diluted and forgotten. For example, in a long paragraph,
the model may forget the subject by the time it reaches the end.

### How LSTMs help with memory
LSTMs have a special "cell state" that acts like a memory lane.
They use gates (forget, input, output) to decide what information
to keep or throw away. This helps them remember important context
over longer sequences compared to simple RNNs.

### What attention solves in sequence-to-sequence tasks
Attention allows the model to look at ALL words in the input at
once and decide which words are most important for each step of
the output. It solves the forgetting problem by giving direct
access to any part of the input, regardless of distance.

### Why transformers are important in modern NLP and Generative AI
Transformers use attention entirely — no recurrence at all.
They process all words in parallel making them much faster to
train. Models like BERT, GPT, and Claude are all transformer
based. They power modern AI tools like ChatGPT, Google Search,
and GitHub Copilot.

In [8]:
import pandas as pd

# model_evaluation.csv
report_dict = classification_report(y_test, y_pred,
              target_names=le.classes_, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv('model_evaluation.csv')
print("model_evaluation.csv saved!")

# sample_predictions.txt
sample = df[['customer_message', 'sentiment_label']].head(10).copy()
sample['predicted'] = le.inverse_transform(model.predict(X_tfidf[:10]))
with open('sample_predictions.txt', 'w') as f:
    f.write(sample.to_string())
print("sample_predictions.txt saved!")

# requirements.txt
with open('requirements.txt', 'w') as f:
    f.write('pandas\nnumpy\nscikit-learn\nnltk\ntensorflow\n')
print("requirements.txt saved!")

model_evaluation.csv saved!
sample_predictions.txt saved!
requirements.txt saved!


In [9]:
from google.colab import files

files.download('model_evaluation.csv')
files.download('sample_predictions.txt')
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>